In [25]:
import requests
from lxml import html
import pandas as pd
import time
import random
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

# recipe_f.csv 파일에서 recipe_id 목록 가져오기 
recipe_ids_df = pd.read_csv('/Users/shindongeun/bigdata_lacture/team_project/mini3/recipe_f.csv')

# 테스트를 위해 처음 10개의 recipe_id만 선택
recipe_ids = recipe_ids_df['recipe_id'].unique()[:5]

# 기존 데이터 불러오기
existing_df = pd.read_csv('all_recipes_1000.csv')
all_recipe_data = existing_df.to_dict('records')

# 요청; 크롬 드라이브 설정
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# 재시도 전략 설정
session = requests.Session()
retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[500, 502, 503, 504],
)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

# 진행 상황 추적
processed_count = 0
total_count = len(recipe_ids)

# 각 recipe_id에 대해 크롤링 수행
for recipe_id in recipe_ids:
    try:
        url = f"https://www.10000recipe.com/recipe/{recipe_id}"
        
        response = session.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'
        
        tree = html.fromstring(response.content)
        
        # 요리 대표 이미지 가져오기
        food_img = tree.xpath('//*[@id="main_thumbs"]/@src')[0] if tree.xpath('//*[@id="main_thumbs"]/@src') else None
        
        # 전체 레시피 스텝을 한번에 가져오기
        steps = tree.xpath('//div[contains(@id, "stepDiv")]/div[contains(@id, "stepdescr")]/text()')
        
        # 스텝 매칭
        for i, step in enumerate(steps, 1):
            step_cleaned = step.strip()
            if step_cleaned:
                recipe_step = {
                    'recipe_id': recipe_id,
                    'recipe_num': i,
                    'recipe_text': step_cleaned,
                    'food_img': food_img if i == 1 else None  # 첫 단계에만 이미지 URL 포함
                }
                all_recipe_data.append(recipe_step)
                print(f"Recipe {recipe_id} - Step {i}: {step_cleaned[:50]}...")  # 각 스텝 출력
        
        processed_count += 1
        print(f"\n✓ 처리 완료: {recipe_id} ({processed_count}/{total_count})\n")
        
        time.sleep(random.uniform(2, 4))
        
    except Exception as e:
        print(f"❌ Error processing recipe {recipe_id}: {str(e)}")
        continue

# 전체 데이터프레임 생성
recipe_df = pd.DataFrame(all_recipe_data)

# CSV 파일로 저장
recipe_df.to_csv('recipe_test.csv', index=False)
print(f"\n✅ 완료! 총 {len(recipe_df)}개의 레시피 단계 저장됨")
print("\n첫 10개의 레시피 데이터:")
print(recipe_df.head(10))

Recipe 128671 - Step 1: 당근과 양파는 깨끗히 씻으신 후에 채썰어 준비한 후 후라이팬에 기름을 두르고 팬을 달군 후...
Recipe 128671 - Step 2: 삶은 당면과 볶은야채를 넣고 잘 섞어주세요.이때 간장1T, 참기름1t 를 넣어서 간을 해주...
Recipe 128671 - Step 3: 나무 젓가락을 이용해 어묵구멍속에 당면과 야채를 어묵을 꾹꾹 눌러가면서 속을 채워주세요....
Recipe 128671 - Step 4: 김을 깔고 그 위에 속을 채운 어묵을 올려놓은 뒤돌돌 말아주세요.녹말물이나 튀김옷 물을 묻...
Recipe 128671 - Step 5: 잘 말은 김말이에 녹말가루를 살짝 묻혀주세요.튀김가루 또는 밀가루를 이용해 튀김옷을 만들어...
Recipe 128671 - Step 6: 잘 달군 팬에 기름을 넉넉히 부어주신 다음얇게 튀김옷을 입힌 김말이를 튀겨주세요....
Recipe 128671 - Step 7: 튀긴 김말이를 키친타올에 올려 불필요한 기름을 빼주세요....

✓ 처리 완료: 128671 (1/5)

Recipe 128892 - Step 1: 당근, 브로콜리, 고추를 잘게 다져주세요....
Recipe 128892 - Step 2: 두부를 칼등으로 으깨주세요....
Recipe 128892 - Step 3: 그 다음 거즈를 이용해 으깬 두부의 물기를 쪼-옥 빼주세요....
Recipe 128892 - Step 4: 으깬 두부와 다져 놓은 야채를 넣고 주물럭 주물럭 잘 섞어주세요. 이때 소금간을 조금 해주...
Recipe 128892 - Step 5: 새우는 머리를 제거하고 이쑤시개로 내장을 제거 한 후에 반으로 갈라서 준비해주세요....
Recipe 128892 - Step 6: 새우를 녹말가루에 묻힌 다음 톡톡톡 털어주세요....
Recipe 128892 - Step 7: 녹말가루를 묻힌 새우를 계란물에 퐁당 담가주세요....
Recipe 128892 - Step 8: 새우를 두부사이에 넣고 토닥

In [6]:
from IPython.display import Image, display, HTML

# HTML 테이블 형태로 이미지 표시
html_content = ""
for url in recipe_df['recipe_img_url']:
    if url:
        html_content += f'<img src="{url}" style="width:300px; margin:10px"/><br>'

display(HTML(html_content))